## 1. 什么是Node
在 LangGraph 里，Node 可以理解为“一个处理步骤”。每个 Node 通常对应一个 Python 函数，它接收当前 `state`，做一些计算，然后返回一个字典，用来更新状态。LangGraph 的状态设计支持对某些字段使用 reducer 聚合，也就是说，节点返回的内容会按规则合并进全局状态。

Node 的核心要求可以概括为下面三点：

- 输入必须是当前状态 `state`，通常写成 `def node_fn(state): ...`。
- 返回值必须是一个字典，字典里的键对应 State 的字段。
- 不要直接修改外部全局状态，推荐“读 state、算结果、返回增量更新”。  

一个直观理解是：Node 不负责“保存整个世界”，它只负责“基于当前状态输出下一步变化”。

## 2. Node 的函数要求

LangGraph 的 Node 函数最重要的是“**输入状态、返回状态更新**”。  
如果你返回的字典里只包含部分字段，图会把这些字段合并回原状态，而不是覆盖整个状态。官方示例也说明了，Node 可以返回类似 `{"foo": "baz"}` 这样的局部更新，然后由图系统负责合并。


建议的 Node 形式：

```python
def my_node(state):     
    # 读取 state 中需要的字段    
    # 执行业务逻辑    
    # 返回一个 dict，表示对 state 的更新    
	return {"some_key": new_value}
```  

如果你的 Node 需要记录中间过程，也可以返回新字段，比如 `trace`、`log`、`step_result`，这样特别适合教学和调试。

## 3. Edge的常用用法
LangGraph中最常见的Edge 用法可以分成三类：  
   - `add_edge`  
   - `add_conditional_edges`  
   - 以及配合 `START/END` 的入口和出口。  
  
---

### 3.1 `add_edge`:
`add_edge(source, target)` 表示从一个节点固定走到另一个节点。比如 `Input → Process`，意味着 Input 执行完后一定会进入 Process。  

它适合最简单、最确定的流程，比如数据清洗、固定顺序的多步处理、线性工作流等。只要前一个节点执行结束，下一节点就会自动运行。

---
  
### 3.2 `add_conditional_edge`
`add_conditional_edges(source, router_fn, mapping)` 表示从某个节点出发，先运行一个“路由函数”，再根据返回结果决定走向哪个节点。官方资料把它用于动态路由，也就是让图根据状态在运行时决定下一步。  

这个机制通常包含三部分：
- `source`：从哪个节点开始分支。    
- `router_fn`：读取 state，返回一个路由标签。
- `mapping`：把路由标签映射到实际节点名，或者映射到 `END`。
  
---

### 3.3 `START / END`  
`START` 和 `END` 是 LangGraph 的虚拟节点。`START` 用于定义入口，`END` 用于定义结束；文档明确提到可以用 `START` 搭配 `add_conditional_edges` 作为条件入口。  

你可以把它们理解成：
- `START`：图从这里“启动”。
- `END`：图执行到这里“停止”。  

这让图结构更清晰，也避免你必须人为定义一个“起始处理节点”。



In [1]:
from typing import Literal, TypedDict
from langgraph.graph import START, END, StateGraph

# 1) 定义状态结构 
#   这里用 TypedDict 来描述整个图的 State。 
#   每个节点都接收这个 state，并返回一个字典来更新它。 
class GraphState(TypedDict):
    input_text: str          # 原始输入文本    
    processed_text: str      # 处理中间结果    
    decision: str            # 判断结果    
    output_text: str         # 最终输出文本    
    step_log: str            # 记录每一步状态变化，便于观察 


In [5]:
# 2) 定义 4 个节点函数 
def input_node(state: GraphState) -> dict:     
	"""
	Input 节点：    
	- 负责初始化输入    
	- 可以理解为图的第一个业务节点    
	"""
    
	print("\n[Input 节点] 进入时 state =", state)     
	# 如果外部没有传入 input_text，就给一个默认值    
	input_text = state.get(
	"input_text", "LangGraph 很适合构建状态流转图"
		)
     
	update = {
		"input_text": input_text,        
		"step_log": f"Input: 收到输入 -> {input_text}"    
	}     
	print("[Input 节点] 返回 update =", update)    
	return update 

def process_node(state: GraphState) -> dict:     
	"""    
	Process 节点：    
	- 读取 input_text    
	- 做简单处理，例如转大写    
	""" 
    
	print("\n[Process 节点] 进入时 state =", state)
	input_text = state["input_text"]  # 这里假设 input_text 已存在    
	processed_text = input_text.upper()     
	update = {
	"processed_text": processed_text,        
	"step_log": state.get("step_log", "") + f" -> Process: 转大写为 {processed_text}"    
	}     
	print("[Process 节点] 返回 update =", update)    
	return update 
	
def decide_node(state: GraphState) -> dict:     
	"""    
	Decide 节点：    
	- 根据 processed_text 做判断    
	- 返回 decision 字段    
	- 注意：这里的“决定”会影响后续 edge 路由   
	"""
    
	print("\n[Decide 节点] 进入时 state =", state)     
	processed_text = state["processed_text"]     
	# 简单判断：如果文本长度超过 12，就认为是 long，否则是 short    
	if len(processed_text) > 12:        
		decision = "long"    
	else:        
		decision = "short"     
	
	update = {        
		"decision": decision,        
		"step_log": state.get("step_log", "") + 
			f" -> Decide: 判断为 {decision}"    
		}     
	print("[Decide 节点] 返回 update =", update)    
	return update 
	
def output_node(state: GraphState) -> dict:     
	"""    
	Output 节点：    
	- 汇总前面所有状态    
	- 生成最终输出    
	"""    
	print("\n[Output 节点] 进入时 state =", state)     
	
	output_text = (        
		f"最终结果：decision={state['decision']}, "
		f"processed_text={state['processed_text']}"    
		)     
	
	update = {        
		"output_text": output_text,        
		"step_log": state.get("step_log", "") + 
			f" -> Output: 生成最终结果"    
			}     
	
	print("[Output 节点] 返回 update =", update)    
	return update 

In [6]:
# 3) 定义条件路由函数 
# 	这个函数会在 Decide 节点之后被调用， 
# 	它根据 state["decision"] 决定下一步走向哪个节点。 

def route_after_decide(state: GraphState) -> Literal["output_node"]:   
	"""    
	条件路由函数：    
	- 读取 decision    
	- 返回下一个节点的标签    
	- 这里因为我们要做一个线性 Graph，      
	  所以无论 decision 是什么，最终都进入 output_node    
	"""    
	decision = state["decision"]    
	print(f"\n[路由函数] 当前 decision = {decision}，下一步进入 output_node")     
	return "output_node" 

In [8]:
# 4) 构建 Graph 

builder = StateGraph(GraphState) # 注册节点
builder.add_node("input_node", input_node)
builder.add_node("process_node", process_node)
builder.add_node("decide_node", decide_node) 
builder.add_node("output_node", output_node)
 
# 固定边：START -> Input -> Process -> Decide 
builder.add_edge(START, "input_node") 
builder.add_edge("input_node", "process_node") 
builder.add_edge("process_node", "decide_node") 

# 条件边：Decide -> Output 
# 这里使用 add_conditional_edges 
# 返回值 "output_node" 会被映射到真正的节点名 output_node
builder.add_conditional_edges(     
	"decide_node",    
	route_after_decide,    
	{        
		"output_node": "output_node"    
	} 
) 

# 固定边：Output -> END 
builder.add_edge("output_node", END) 

# 编译图 
graph = builder.compile() 

In [10]:
# 5) 运行 Graph 
# 	传入初始状态。你可以修改 input_text 来观察 State 如何变化。

initial_state = {     
	"input_text": "Hello LangGraph"  # 尝试输入'hi'，观察变化 
	}

print("\n========== 开始执行 Graph ==========") 
final_state = graph.invoke(initial_state) 
print("========== Graph 执行结束 ==========\n") 
print("最终返回的 state =") 
print(final_state) 


========== 开始执行 Graph ==========

[Input 节点] 进入时 state = {'input_text': 'Hi'}
[Input 节点] 返回 update = {'input_text': 'Hi', 'step_log': 'Input: 收到输入 -> Hi'}

[Process 节点] 进入时 state = {'input_text': 'Hi', 'step_log': 'Input: 收到输入 -> Hi'}
[Process 节点] 返回 update = {'processed_text': 'HI', 'step_log': 'Input: 收到输入 -> Hi -> Process: 转大写为 HI'}

[Decide 节点] 进入时 state = {'input_text': 'Hi', 'processed_text': 'HI', 'step_log': 'Input: 收到输入 -> Hi -> Process: 转大写为 HI'}
[Decide 节点] 返回 update = {'decision': 'short', 'step_log': 'Input: 收到输入 -> Hi -> Process: 转大写为 HI -> Decide: 判断为 short'}

[路由函数] 当前 decision = short，下一步进入 output_node

[Output 节点] 进入时 state = {'input_text': 'Hi', 'processed_text': 'HI', 'decision': 'short', 'step_log': 'Input: 收到输入 -> Hi -> Process: 转大写为 HI -> Decide: 判断为 short'}
[Output 节点] 返回 update = {'output_text': '最终结果：decision=short, processed_text=HI', 'step_log': 'Input: 收到输入 -> Hi -> Process: 转大写为 HI -> Decide: 判断为 short -> Output: 生成最终结果'}
========== Graph 执行结束 ==========
